In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# --- Configuration ---
IMAGE_PATH = "assets/sample_image.jpg"   # any image works — just point this at a file
RESIZE_DIM = (256, 256)
TOP_K_SINGULAR_VALUES = 40

# --- Load and preprocess image ---
img = Image.open(IMAGE_PATH).convert('L').resize(RESIZE_DIM)
image_matrix = np.array(img, dtype=np.float64) / 255.0

plt.figure(figsize=(4, 4))
plt.imshow(image_matrix, cmap='gray')
plt.title('Preprocessed Grayscale Image')
plt.axis('off')
plt.show()

In [7]:
# --- Step 1: Power iteration and deflation for SVD --- #

def power_iteration(matrix, num_iterations=1000, tol=1e-8):
    """Return the dominant eigenvalue and eigenvector of a symmetric matrix
    via power iteration."""
    n = matrix.shape[0]
    b_k = np.random.rand(n)
    b_k /= np.linalg.norm(b_k)

    for _ in range(num_iterations):
        b_k1 = matrix @ b_k
        b_k_new = b_k1 / np.linalg.norm(b_k1)

        if np.linalg.norm(b_k_new - b_k) < tol:
            b_k = b_k_new
            break
        b_k = b_k_new

    eigenvalue = b_k.T @ matrix @ b_k / (b_k.T @ b_k)
    return eigenvalue, b_k


def svd_from_scratch(A, k, num_iterations=1000, tol=1e-8):
    """Compute the top-k singular values/vectors of A via power iteration
    with deflation.

    Parameters
    ----------
    A : (m, n) ndarray
        Input matrix.
    k : int
        Number of singular values/vectors to compute.

    Returns
    -------
    U_scratch : (m, k) ndarray
    singular_values : (k,) ndarray
    V_scratch : (n, k) ndarray
    """
    m, n = A.shape

    ATA = A.T @ A
    # AAT = A @ A.T

    singular_values_list = []
    V_scratch = np.zeros((n, k))
    U_scratch = np.zeros((m, k))

    # Step 2: Extract eigenvectors/eigenvalues of A.T @ A via deflation
    current_matrix_for_v = np.copy(ATA)

    for i in range(k):
        eigenvalue_v, eigenvector_v = power_iteration(current_matrix_for_v, num_iterations, tol)

        sigma_i = np.sqrt(abs(eigenvalue_v))
        singular_values_list.append(sigma_i)
        V_scratch[:, i] = eigenvector_v

        # Deflation: remove this eigenpair's contribution before the next iteration
        current_matrix_for_v -= eigenvalue_v * np.outer(eigenvector_v, eigenvector_v)

    # Step 3: Derive U from A, V, and the singular values: u_i = A @ v_i / sigma_i
    for i in range(k):
        if singular_values_list[i] > 1e-9:  # avoid division by ~0
            U_scratch[:, i] = (A @ V_scratch[:, i]) / singular_values_list[i]
        else:
            U_scratch[:, i] = np.zeros(m)

    # Sort by descending singular value
    sorted_indices = np.argsort(singular_values_list)[::-1]
    singular_values_sorted = np.array(singular_values_list)[sorted_indices]
    U_scratch = U_scratch[:, sorted_indices]
    V_scratch = V_scratch[:, sorted_indices]

    return U_scratch, np.diag(singular_values_sorted), V_scratch.T

In [8]:
# --- Perform SVD from scratch and reconstruct the image --- #
U_scratch, S_scratch, Vt_scratch = svd_from_scratch(image_matrix, TOP_K_SINGULAR_VALUES)
singular_values_for_plot = np.diag(S_scratch)

reconstructed_matrix_scratch = U_scratch @ S_scratch @ Vt_scratch
reconstructed_matrix_scratch = np.clip(reconstructed_matrix_scratch, 0, 1)

print(f"Original image shape:      {image_matrix.shape}")
print(f"U_scratch shape:           {U_scratch.shape}")
print(f"V_scratch shape:           {Vt_scratch.T.shape}")
print(f"Reconstructed shape:       {reconstructed_matrix_scratch.shape}")
print(f"Top {TOP_K_SINGULAR_VALUES} singular values: {singular_values_for_plot[:5]}")

# --- Reconstruction error (MSE) and % variance retained --- #
mse_scratch = np.mean((image_matrix - reconstructed_matrix_scratch) ** 2)

# Key identity: sum of ALL singular values squared = ||A||_F^2 (Frobenius norm squared)
#             = sum over every pixel of (pixel value)^2 = trace(A^T A)
# So the total variance (denominator) only needs the original image, not a full SVD.
total_variance_scratch = np.sum(image_matrix ** 2)
retained_variance_scratch = np.sum(singular_values_for_plot ** 2)
variance_retained_scratch = (retained_variance_scratch / total_variance_scratch) * 100

print(f"MSE (scratch SVD, k={TOP_K_SINGULAR_VALUES}): {mse_scratch:.6f}")
print(f"Variance retained (scratch SVD, k={TOP_K_SINGULAR_VALUES}): {variance_retained_scratch:.2f}%")



MSE (scratch SVD, k=40): 0.000339
Variance retained (scratch SVD, k=40): 99.87%


In [ ]:
# --- Plot original vs. reconstructed image, and singular value spectrum --- #
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(image_matrix, cmap='gray')
plt.title('Original Image')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(reconstructed_matrix_scratch, cmap='gray')
plt.title(f'Reconstructed Image (Scratch SVD, k={TOP_K_SINGULAR_VALUES})')
plt.axis('off')
plt.suptitle(f'Variance Retained: {variance_retained_scratch:.2f}%', fontsize=14)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.semilogy(np.arange(1, len(singular_values_for_plot) + 1), singular_values_for_plot, '-o')
plt.title('Singular Value Magnitudes (Scratch SVD, Log Scale)')
plt.xlabel('Singular Value Index')
plt.ylabel('Singular Value Magnitude (log scale)')
plt.grid(True, which='both')
plt.show()

In [10]:
# ============================================================ #
# ==================  USING TENSORFLOW  ===================== #
# ============================================================ #
# Uses the same preprocessed `image_matrix` and TOP_K_SINGULAR_VALUES
# from the scratch section above; no data loading/preprocessing repeated.

import tensorflow as tf

image_tensor = tf.constant(image_matrix, dtype=tf.float64)

# --- SVD via tf.linalg.svd() --- #
# tf.linalg.svd returns (singular_values, U, V), where V is the V matrix, not V.T
s, U_tf, V_tf = tf.linalg.svd(image_tensor)

s_np = s.numpy()
U_tf_np = U_tf.numpy()
Vt_tf_np = V_tf.numpy().T

# --- Reconstruct the image from the top-k components --- #
U_k_tf = U_tf_np[:, :TOP_K_SINGULAR_VALUES]
S_k_tf = np.diag(s_np[:TOP_K_SINGULAR_VALUES])
Vt_k_tf = Vt_tf_np[:TOP_K_SINGULAR_VALUES, :]

reconstructed_matrix_tf = U_k_tf @ S_k_tf @ Vt_k_tf
reconstructed_matrix_tf = np.clip(reconstructed_matrix_tf, 0, 1)

print(f"Original image shape:      {image_matrix.shape}")
print(f"U_tf shape:                {U_k_tf.shape}")
print(f"V_tf shape:                {Vt_k_tf.shape}")
print(f"Reconstructed shape:       {reconstructed_matrix_tf.shape}")
print(f"Top {TOP_K_SINGULAR_VALUES} singular values: {s_np[:5]}")

# --- Reconstruction error (MSE) and % variance retained --- #
mse_tf = np.mean((image_matrix - reconstructed_matrix_tf) ** 2)

total_variance_tf = np.sum(s_np ** 2)
variance_retained_tf = np.sum(s_np[:TOP_K_SINGULAR_VALUES] ** 2) / total_variance_tf

print(f"MSE (TensorFlow SVD, k={TOP_K_SINGULAR_VALUES}): {mse_tf:.6f}")
print(f"Variance retained (TensorFlow SVD, k={TOP_K_SINGULAR_VALUES}): {variance_retained_tf * 100:.2f}%")


MSE (TensorFlow SVD, k=40): 0.000339
Variance retained (TensorFlow SVD, k=40): 99.87%


In [ ]:
# --- Plot original vs. reconstructed image, and singular value spectrum --- #
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(image_matrix, cmap='gray')
plt.title('Original Image')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(reconstructed_matrix_tf, cmap='gray')
plt.title(f'Reconstructed Image (TensorFlow SVD, k={TOP_K_SINGULAR_VALUES})')
plt.axis('off')
plt.suptitle(f'Variance Retained: {variance_retained_tf * 100:.2f}%', fontsize=14)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.semilogy(np.arange(1, len(s_np) + 1), s_np, '-o')
plt.title('Singular Value Magnitudes (TensorFlow SVD, Log Scale)')
plt.xlabel('Singular Value Index')
plt.ylabel('Singular Value Magnitude (log scale)')
plt.grid(True, which='both')
plt.show()